# Fenrir: PCA orientado a clustering
Esta versión deja de fijar el PCA en 2 componentes desde el principio y pasa a seleccionar el espacio reducido por varianza explicada acumulada.
Además compara varios escalados y varios algoritmos de clustering usando métricas complementarias, y expone diagnósticos útiles del PCA: scree plot, círculo de correlaciones, contribuciones y calidad de representación (`cos2`).

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

candidate_project_roots = [
    Path.cwd(),
    Path.cwd() / "fenrir",
    Path.cwd().parent,
    Path.cwd().parent / "fenrir",
]

try:
    import fenrir as fenrir_package
except ModuleNotFoundError:
    FENRIR_PROJECT_ROOT = next(
        (
            path
            for path in candidate_project_roots
            if (path / "core.py").exists() and (path / "__init__.py").exists()
        ),
        None,
    )
    if FENRIR_PROJECT_ROOT is None:
        raise FileNotFoundError(
            "No se encontro fenrir instalado ni una copia local del proyecto. "
            "Instala la libreria o coloca este notebook junto al repo."
        )

    package_parent = FENRIR_PROJECT_ROOT.parent
    if str(package_parent) not in sys.path:
        sys.path.insert(0, str(package_parent))

    import fenrir as fenrir_package

fenrir_package = importlib.reload(fenrir_package)
Fenrir = fenrir_package.Fenrir
package_file = getattr(fenrir_package, "__file__", None)
if package_file is None:
    raise RuntimeError("No se pudo resolver la ruta del paquete fenrir cargado.")

FENRIR_PROJECT_ROOT = Path(package_file).resolve().parent
FENRIR_EXAMPLES_DIR = FENRIR_PROJECT_ROOT / "examples"

print(f"Fenrir importado desde: {FENRIR_PROJECT_ROOT}")

## Bootstrap del paquete
Este notebook intenta importar primero `fenrir` desde el entorno activo. Si no lo encuentra, busca una copia local del proyecto junto al notebook y evita depender de rutas absolutas de Windows.

## Demo con Titanic
Este ejemplo busca primero `Titanic-Dataset.csv` junto al notebook, en `examples/` o en `data/`. Si no lo encuentra, intenta cargar Titanic desde OpenML. Y si tampoco hay red, cae a un dataset **sintetico** con la misma forma que Titanic, para que la plantilla siga siendo ejecutable en un clon recien bajado: la celda avisa por pantalla y `data_source` lo deja por escrito. Esos datos no son reales y no sirven para sacar conclusiones, solo para recorrer el flujo antes de enchufar el tuyo.
La variable objetivo se declara con `target="Survived"`. Fenrir la separa del PCA y del clustering, y la usa despues para perfilar los clusters, estratificar la evaluacion holdout y compararlos con una referencia externa.
La libreria no se adapta ad hoc a Titanic: lo que cambia es el modo de uso. Ahora expone tambien una lista libre de variables influyentes, un dataframe con la columna de cluster, un `groupby` por cluster y una busqueda del subconjunto mas compacto que mejora el clustering.

In [ ]:
from pathlib import Path

candidate_csv_paths = [
    Path.cwd() / "Titanic-Dataset.csv",
    Path.cwd() / "examples" / "Titanic-Dataset.csv",
    Path.cwd() / "data" / "Titanic-Dataset.csv",
    Path.cwd().parent / "Titanic-Dataset.csv",
    Path.cwd().parent / "examples" / "Titanic-Dataset.csv",
    Path.cwd().parent / "data" / "Titanic-Dataset.csv",
]

if "FENRIR_PROJECT_ROOT" in globals():
    candidate_csv_paths.extend(
        [
            FENRIR_PROJECT_ROOT.parent / "Titanic-Dataset.csv",
            FENRIR_PROJECT_ROOT / "Titanic-Dataset.csv",
            FENRIR_PROJECT_ROOT / "data" / "Titanic-Dataset.csv",
        ]
    )

if "FENRIR_EXAMPLES_DIR" in globals():
    candidate_csv_paths.extend(
        [
            FENRIR_EXAMPLES_DIR / "Titanic-Dataset.csv",
            FENRIR_EXAMPLES_DIR / "data" / "Titanic-Dataset.csv",
        ]
    )

csv_path = next((path for path in candidate_csv_paths if path.exists()), None)
if csv_path is not None:
    demo = pd.read_csv(csv_path)
    data_source = f"CSV local: {csv_path}"
else:
    try:
        from sklearn.datasets import fetch_openml
    except ImportError as exc:
        raise FileNotFoundError(
            "No se encontro 'Titanic-Dataset.csv' y tampoco esta disponible "
            "fetch_openml para descargar el dataset de respaldo."
        ) from exc

    try:
        openml_dataset = fetch_openml("titanic", version=1, as_frame=True)
    except Exception:
        # Tercer plan: sin CSV local y sin red, la plantilla sigue siendo
        # ejecutable con un dataset sintetico de la misma forma que Titanic.
        # No son datos reales y el notebook lo deja dicho en `data_source`:
        # sirve para recorrer el flujo, no para sacar conclusiones.
        openml_dataset = None

if csv_path is None and openml_dataset is None:
    rng = np.random.default_rng(42)
    n_rows = 400
    pclass = rng.choice([1, 2, 3], size=n_rows, p=[0.22, 0.21, 0.57])
    sex = rng.choice(["male", "female"], size=n_rows, p=[0.64, 0.36])
    age = np.clip(rng.normal(np.where(pclass == 1, 38, np.where(pclass == 2, 30, 25)), 12), 0.5, 80).round(1)
    sibsp = rng.poisson(np.where(pclass == 3, 0.6, 0.35), size=n_rows)
    parch = rng.poisson(0.4, size=n_rows)
    fare = (rng.gamma(2.0, np.where(pclass == 1, 40, np.where(pclass == 2, 13, 7))) + 4).round(2)
    embarked = rng.choice(["S", "C", "Q"], size=n_rows, p=[0.72, 0.19, 0.09])

    survival_score = (
        1.9 * (sex == "female")
        + 0.7 * (pclass == 1)
        + 0.2 * (pclass == 2)
        - 0.015 * age
        - 0.9
    )
    survived = (rng.random(n_rows) < 1 / (1 + np.exp(-survival_score))).astype(int)

    demo = pd.DataFrame(
        {
            "PassengerId": np.arange(1, n_rows + 1),
            "Survived": survived,
            "Pclass": pclass,
            "Name": [f"Pasajero sintetico {index:03d}" for index in range(1, n_rows + 1)],
            "Sex": sex,
            "Age": age,
            "SibSp": sibsp,
            "Parch": parch,
            "Ticket": [f"SYN-{rng.integers(10000, 99999)}" for _ in range(n_rows)],
            "Fare": fare,
            "Cabin": np.where(pclass == 1, [f"C{value}" for value in rng.integers(1, 120, n_rows)], None),
            "Embarked": embarked,
        }
    )
    data_source = "Titanic SINTETICO (sin CSV local y sin acceso a OpenML)"
    print("AVISO: no son datos reales. Sustituye `demo` por tu propio DataFrame.")

elif csv_path is None:
    demo = openml_dataset.frame.copy()
    demo = demo.rename(
        columns={
            "survived": "Survived",
            "pclass": "Pclass",
            "name": "Name",
            "sex": "Sex",
            "age": "Age",
            "sibsp": "SibSp",
            "parch": "Parch",
            "ticket": "Ticket",
            "fare": "Fare",
            "cabin": "Cabin",
            "embarked": "Embarked",
        }
    )
    if "PassengerId" not in demo.columns:
        demo.insert(0, "PassengerId", range(1, len(demo) + 1))

    preferred_columns = [
        "PassengerId",
        "Survived",
        "Pclass",
        "Name",
        "Sex",
        "Age",
        "SibSp",
        "Parch",
        "Ticket",
        "Fare",
        "Cabin",
        "Embarked",
    ]
    demo = demo.loc[:, [column for column in preferred_columns if column in demo.columns]].copy()
    demo["Survived"] = pd.to_numeric(demo["Survived"], errors="coerce")
    data_source = "OpenML Titanic fallback"

print(f"Dataset cargado desde: {data_source}")
print(f"Forma del dataset: {demo.shape}")
display(demo.head())

In [ ]:
fenrir_config = {
    "target": "Survived",
    "exclude_columns": ["PassengerId", "Name", "Ticket", "Cabin"],
    "variance_threshold": 0.9,
    "cluster_range": range(2, 8),
    "algorithms": ("kmeans", "gmm", "agglomerative"),
    "max_categories": 12,
    "random_state": 42,
}

analysis_config = {
    "top_n": 5,
    "leaderboard_top_n": 6,
    "holdout_top_n": 5,
    "compact_max_features": 5,
    "compact_criterion": "holdout_silhouette",
    "compact_include_holdout": None,
    "test_size": 0.25,
    "n_splits": 2,
    "stratify_target": True,
    "min_cluster_share": 0.01,
    "include_stability": True,
    "stability_sample_fraction": 0.8,
    "stability_n_splits": 3,
    "stability_top_n": 5,
    "stability_stratify_target": True,
}

plot_config = {
    "figsize": (18, 12),
    "correlation_top_n": 10,
    "contribution_top_n": 8,
    "quality_top_n": 8,
    "projection_components": 2,
}

dashboard_config = {
    "metric_rows": [
        "explained_variance",
        "variance_first_two",
        "silhouette",
        "davies_bouldin",
        "ari",
        "nmi",
        "v_measure",
    ],
    "holdout_rows": 5,
    "compact_rows": 5,
    "stability_rows": 5,
}

parameter_preview = pd.Series(
    {
        "target": fenrir_config["target"],
        "cluster_range": list(fenrir_config["cluster_range"]),
        "compact_criterion": analysis_config["compact_criterion"],
        "n_splits": analysis_config["n_splits"],
        "min_cluster_share": analysis_config["min_cluster_share"],
        "include_stability": analysis_config["include_stability"],
        "stability_sample_fraction": analysis_config["stability_sample_fraction"],
        "stability_n_splits": analysis_config["stability_n_splits"],
    },
    name="valor",
).to_frame()

print("Ajusta aqui los parametros principales de Fenrir")
display(parameter_preview)

## Parametros de usuario
Ajusta esta celda antes de reejecutar la demo si quieres cambiar el rango de clusters, la busqueda compacta o la validacion holdout sin tocar el resto del notebook.

In [ ]:
fenrir = Fenrir(demo, **fenrir_config)
clusters = fenrir.fit_predict()
clustered_demo = fenrir.clustered_dataframe(
    include_components=True,
    n_components=plot_config["projection_components"],
)
metric_view = fenrir.metric_report().loc[:, ["metric", "value", "acceptable", "interpretation"]]
influential_list = fenrir.list_influential_variables(top_n=max(8, analysis_config["top_n"]))
cluster_size_view = fenrir.cluster_size_report()
cluster_group_view = fenrir.cluster_groupby(top_n=analysis_config["top_n"])

print("Mejor configuracion")
display(fenrir.best_configuration().rename("valor").to_frame())

print("Lectura orientativa de metricas")
display(metric_view)

print("Resumen del preprocesado")
display(fenrir.preprocessing_report())

print("Comparativa por escalado")
display(fenrir.scaler_report())

print("Top del leaderboard")
display(fenrir.summary(top_n=analysis_config["leaderboard_top_n"]))

print("Lista libre de variables mas influyentes")
print(influential_list)

print("Tamano de clusters")
display(cluster_size_view)

print("Groupby por cluster")
display(cluster_group_view)

print("Relacion cluster-objetivo")
display(fenrir.cluster_target_report(normalize="index"))

print("Dataframe con cluster anadido")
display(clustered_demo.head())

## Workbench interactivo

Define la variable objetivo, reajusta Fenrir, lanza validacion profunda, exporta tablas y graficos, guarda presets persistentes en JSON, genera un Excel ejecutivo multihja y alterna entre vista completa, solo PCA, solo clustering o comparacion lado a lado de presets.

In [ ]:
fenrir_workbench = fenrir.interactive_workbench(
    fenrir_config=fenrir_config,
    analysis_config=analysis_config,
)
fenrir_tables = fenrir_workbench

display(fenrir_workbench)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=plot_config["figsize"])
fenrir.plot_scree(ax=axes[0, 0])
fenrir.plot_correlation_circle(ax=axes[0, 1], top_n=plot_config["correlation_top_n"])
fenrir.plot_feature_contributions(ax=axes[1, 0], top_n=plot_config["contribution_top_n"])
fenrir.plot_feature_quality(ax=axes[1, 1], top_n=plot_config["quality_top_n"])
plt.tight_layout()
plt.show()

fenrir.plot_clusters()
plt.show()

In [ ]:
report = fenrir.analysis_report(**analysis_config)

best = report["best_configuration"]
embedding = fenrir.transform()
source_report = fenrir.source_feature_report(top_n=plot_config["contribution_top_n"])
top_variables = report["influential_variables"]
selected_dataset = fenrir.influential_dataset(top_n=analysis_config["top_n"], include_target=True)
target_profile = report["cluster_target_report"]
holdout_summary = report["holdout_summary"]
holdout_top = report["holdout_top"]
holdout_view = holdout_top.loc[:, [
    column
    for column in [
        "scaler_name",
        "algorithm",
        "n_clusters",
        "n_components",
        "test_silhouette_mean",
        "test_davies_bouldin_mean",
        "test_v_measure_mean",
        "test_nmi_mean",
    ]
    if column in holdout_top.columns
]]
subset_search = report["compact_subset_search"]
subset_view = subset_search.loc[:, [
    column
    for column in [
        "n_selected_features",
        "selected_variables",
        "test_silhouette_mean",
        "test_davies_bouldin_mean",
        "silhouette",
        "explained_variance",
        "davies_bouldin",
        "min_cluster_size",
        "min_cluster_share",
        "selection_status",
    ]
    if column in subset_search.columns
]]
best_compact_variables = report["best_compact_variables"]
fenrir_compact = fenrir.best_influential_model()
compact_best = report["best_compact_configuration"]
compact_group_view = report["best_compact_groupby"]
stability_summary = report["stability_summary"]
stability_top = report["stability_top"]
stability_view = stability_top.loc[:, [
    column
    for column in [
        "split",
        "sample_size",
        "ari_vs_base",
        "nmi_vs_base",
        "v_measure_vs_base",
        "same_algorithm",
        "same_n_clusters",
        "same_n_components",
        "min_cluster_size",
    ]
    if column in stability_top.columns
]] if stability_top is not None else pd.DataFrame()

assert len(clusters) == len(demo)
assert embedding.shape[1] == int(best["n_components"])
assert best["explained_variance"] >= fenrir.variance_threshold
assert np.isfinite(float(best["silhouette"]))
assert len(top_variables) == analysis_config["top_n"]
assert set(top_variables).issubset(demo.columns)
assert set(top_variables).issubset(selected_dataset.columns)
assert fenrir.target_name_ == fenrir_config["target"]
assert {"weighted_contribution", "weighted_cos2"}.issubset(source_report.columns)
assert target_profile is not None and target_profile.shape[0] >= 2
assert "cluster" in clustered_demo.columns
assert not holdout_summary.empty
assert {"test_silhouette_mean", "test_davies_bouldin_mean"}.issubset(holdout_top.columns)
assert not subset_search.empty
assert {"selection_status"}.issubset(subset_search.columns)
assert subset_search.iloc[0]["selection_status"] == "valido"
assert len(best_compact_variables) >= 2
assert set(best_compact_variables).issubset(demo.columns)
assert len(fenrir_compact.predict()) == len(demo)
assert compact_best["explained_variance"] >= fenrir_compact.variance_threshold
assert np.isfinite(float(compact_best["silhouette"]))
assert stability_summary is not None and not stability_summary.empty
assert stability_top is not None and not stability_top.empty
assert {"ari_vs_base_mean", "same_configuration_rate", "stability_reading"}.issubset(stability_summary.columns)

print("Top holdout estratificado")
display(holdout_view)

print("Busqueda compacta guiada por holdout")
display(subset_view)

print("Lista compacta accesible para el usuario:", best_compact_variables)
print("Configuracion del mejor subconjunto compacto")
display(compact_best.rename("valor").to_frame())

print("Estabilidad por remuestreo")
display(stability_summary)
display(stability_view)

print("Groupby del mejor subconjunto compacto")
display(compact_group_view)

In [ ]:
if "report" not in globals() or not isinstance(report, dict):
    report = fenrir.analysis_report(**analysis_config)

dashboard_parameters = pd.Series(
    {
        "cluster_range": list(fenrir_config["cluster_range"]),
        "compact_criterion": analysis_config["compact_criterion"],
        "n_splits": analysis_config["n_splits"],
        "min_cluster_share": analysis_config["min_cluster_share"],
        "include_stability": analysis_config["include_stability"],
        "stability_sample_fraction": analysis_config["stability_sample_fraction"],
        "stability_n_splits": analysis_config["stability_n_splits"],
    },
    name="valor",
).to_frame()

dashboard_metrics = report["metric_report"].copy()
dashboard_metrics = dashboard_metrics[
    dashboard_metrics["metric"].isin(dashboard_config["metric_rows"])
].reset_index(drop=True)

dashboard_holdout = report["holdout_top"].head(dashboard_config["holdout_rows"])
dashboard_holdout = dashboard_holdout.loc[:, [
    column
    for column in [
        "scaler_name",
        "algorithm",
        "n_clusters",
        "n_components",
        "test_silhouette_mean",
        "test_davies_bouldin_mean",
        "test_v_measure_mean",
    ]
    if column in dashboard_holdout.columns
]]

dashboard_compact = report["compact_subset_search"].head(dashboard_config["compact_rows"])
dashboard_compact = dashboard_compact.loc[:, [
    column
    for column in [
        "n_selected_features",
        "selected_variables",
        "test_silhouette_mean",
        "test_davies_bouldin_mean",
        "silhouette",
        "min_cluster_size",
        "selection_status",
    ]
    if column in dashboard_compact.columns
]]

dashboard_stability_summary = report["stability_summary"]
dashboard_stability_top = report["stability_top"].head(dashboard_config["stability_rows"])
dashboard_stability_top = dashboard_stability_top.loc[:, [
    column
    for column in [
        "split",
        "sample_size",
        "ari_vs_base",
        "nmi_vs_base",
        "same_algorithm",
        "same_n_clusters",
        "same_n_components",
        "min_cluster_size",
    ]
    if column in dashboard_stability_top.columns
]]

print("Resumen ejecutivo de Fenrir")
print("Variables influyentes libres:", report["influential_variables"])
print("Mejor lista compacta:", report["best_compact_variables"])

print("Parametros activos")
display(dashboard_parameters)

print("Mejor configuracion base")
display(report["best_configuration"].rename("valor").to_frame())

print("Metricas clave")
display(dashboard_metrics)

print("Top holdout")
display(dashboard_holdout)

print("Top subconjuntos compactos")
display(dashboard_compact)

print("Resumen de estabilidad")
display(dashboard_stability_summary)

print("Splits mas estables")
display(dashboard_stability_top)

print("Resumen por cluster del modelo base")
display(report["cluster_groupby"])

print("Resumen por cluster del mejor modelo compacto")
display(report["best_compact_groupby"])

## Panel resumido y estabilidad
Esta celda final consume `analysis_report()` y deja una vista corta con parametros, metrica base, holdout, estabilidad por remuestreo y mejor subconjunto compacto.